# Lab Sheet 1: Sentiment Analysis Using Traditional Machine Learning

**Practical:** Text preprocessing, Bag-of-Words, and a traditional ML sentiment classifier

**Tools:** Python, Pandas, NumPy, NLTK, Scikit-learn, Matplotlib, Seaborn

**Dataset:** [Sentiment Labelled Sentences](https://archive.ics.uci.edu/dataset/331/sentiment%2Blabelled%2Bsentences) (UCI) — short Amazon, IMDb, and Yelp reviews labelled positive (1) or negative (0)

This notebook has not been run — run it top to bottom locally or in Colab.

In [ ]:
import re
import string
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download("stopwords")

## Step 1 — Download and load the dataset

In [ ]:
DATA_DIR = Path("sentiment_labelled_sentences")
ZIP_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
ZIP_PATH = Path("sentiment_labelled_sentences.zip")

if not DATA_DIR.exists():
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(".")

SOURCE_DIR = Path("sentiment labelled sentences")

files = {
    "amazon": SOURCE_DIR / "amazon_cells_labelled.txt",
    "imdb": SOURCE_DIR / "imdb_labelled.txt",
    "yelp": SOURCE_DIR / "yelp_labelled.txt",
}

frames = []
for source, path in files.items():
    df = pd.read_csv(path, sep="\t", header=None, names=["text", "label"])
    df["source"] = source
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print("Total reviews:", len(data))
data.head()

## Step 2 — Basic text preprocessing

Lowercase, strip punctuation/digits, and collapse extra whitespace.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["clean_text"] = data["text"].apply(clean_text)

# Compare a few original vs. preprocessed examples
for original, cleaned in zip(data["text"].head(5), data["clean_text"].head(5)):
    print("Original: ", original)
    print("Cleaned:  ", cleaned)
    print()

## Step 3 — Explore the text data

In [ ]:
print("Positive examples:")
print(data[data["label"] == 1]["text"].head(3).to_string(index=False))

print("\nNegative examples:")
print(data[data["label"] == 0]["text"].head(3).to_string(index=False))

print("\nLabel counts:")
print(data["label"].value_counts())

sns.countplot(x="label", data=data)
plt.title("Positive (1) vs. Negative (0) review counts")
plt.show()

In [ ]:
stop_words = set(stopwords.words("english"))

def top_words(label, n=15):
    words = " ".join(data[data["label"] == label]["clean_text"]).split()
    words = [w for w in words if w not in stop_words]
    return Counter(words).most_common(n)

print("Most frequent words — positive reviews:")
print(top_words(1))

print("\nMost frequent words — negative reviews:")
print(top_words(0))

## Step 4 — Split the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

## Step 5 — Create a Bag-of-Words representation

In [ ]:
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Train BoW matrix shape:", X_train_bow.shape)

## Step 6 — Train a machine learning model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)

## Step 7 — Evaluate the model's performance

In [ ]:
y_pred = model.predict(X_test_bow)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"],
            yticklabels=["negative", "positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Step 8 — Test the model with new sentences

In [ ]:
new_sentences = [
    "This product completely exceeded my expectations, I love it!",
    "Terrible service and the food was cold, never coming back.",
    "It was okay, nothing special but not bad either.",
]

new_clean = [clean_text(s) for s in new_sentences]
new_bow = vectorizer.transform(new_clean)
predictions = model.predict(new_bow)

for sentence, pred in zip(new_sentences, predictions):
    label = "positive" if pred == 1 else "negative"
    print(f"{label:>8}  —  {sentence}")